# Data Warehousing and Business Intelligence Coursework - Group 6
## Anahita Hessami (Part A) - Morteza Nateghi (Part B) - Kamran Rzayev (Part C)
---

### This report contains two parts:
*   #### Coding & Results
*   #### Visualizations

# Coding
---

In [20]:
# Install and import required packages
!pip install yfinance yahoofinancials pandas numpy matplotlib seaborn scipy -q

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from yahoofinancials import YahooFinancials
from datetime import datetime

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})


## Part A

In [21]:
# Part A-1
# Extract S&P 500 Tickers from Wikipedia

SP500_WIKI_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {'User-Agent': 'Mozilla/5.0'}
sp500_info = pd.read_html(SP500_WIKI_URL, storage_options=headers)[0]

sp500_info.rename(columns={
    "Symbol": "Ticker",
    "Security": "Company",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "Sub_Industry"
}, inplace=True)

# Yahoo Finance uses '-' not '.' (e.g. BRK.B -> BRK-B)
sp500_info["Ticker"] = sp500_info["Ticker"].str.replace(".", "-", regex=False)
tickers = sp500_info["Ticker"].tolist()

print(f"{len(tickers)} tickers extracted.")
print(sp500_info[["Ticker", "Company", "Sector", "Headquarters Location", "Founded"]].head(10).to_string(index=False))

503 tickers extracted.
Ticker                Company                 Sector   Headquarters Location     Founded
   MMM                     3M            Industrials   Saint Paul, Minnesota        1902
   AOS            A. O. Smith            Industrials    Milwaukee, Wisconsin        1916
   ABT    Abbott Laboratories            Health Care North Chicago, Illinois        1888
  ABBV                 AbbVie            Health Care North Chicago, Illinois 2013 (1888)
   ACN              Accenture Information Technology         Dublin, Ireland        1989
  ADBE             Adobe Inc. Information Technology    San Jose, California        1982
   AMD Advanced Micro Devices Information Technology Santa Clara, California        1969
   AES        AES Corporation              Utilities     Arlington, Virginia        1981
   AFL                  Aflac             Financials       Columbus, Georgia        1955
     A   Agilent Technologies            Health Care Santa Clara, California        199

In [22]:
# Part A-2
# Load Daily Market Data with yfinance

START_DATE = "2022-01-01"
END_DATE   = "2025-01-01"

print(f" Downloading daily Close prices via yfinance …")
print(f"    Period  : {START_DATE} → {END_DATE}")
print(f"    Tickers : {len(tickers)}")
print()

raw_prices = yf.download(
    tickers     = tickers,
    start       = START_DATE,
    end         = END_DATE,
    auto_adjust = True,
    progress    = True,
    threads     = True
)["Close"]

# ── Transpose: rows = tickers, columns = trading days ────────────────────────
raw_prices = raw_prices.T
print(f"\n yfinance download complete.")
print(f"  Shape after transpose : {raw_prices.shape}  (rows=tickers, cols=trading days)")
print()
print("First 5 rows (tickers) × first 5 trading days:")
print(raw_prices.iloc[:5, :5].round(2))

    Period  : 2022-01-01 → 2025-01-01
    Tickers : 503



[*********************100%***********************]  503 of 503 completed
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['Q', 'SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2022-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1641013200, endDate = 1735707600")')



 yfinance download complete.
  Shape after transpose : (503, 753)  (rows=tickers, cols=trading days)

First 5 rows (tickers) × first 5 trading days:
Date    2022-01-03  2022-01-04  2022-01-05  2022-01-06  2022-01-07
Ticker                                                            
A           151.70      146.57      144.06      144.57      140.72
AAPL        178.10      175.84      171.17      168.31      168.47
ABBV        114.89      114.67      115.28      114.73      114.44
ABNB        172.68      170.80      162.25      159.75      166.05
ABT         127.55      124.55      123.99      123.97      124.35


In [23]:
# Part A-3
# Identify and Remove Invalid Data
# Work on a copy; transpose back to (dates × tickers) for cleaning, then re-transpose so the cleaned result stays (tickers × dates).

prices = raw_prices.T.copy()    # (trading days × tickers) for cleaning operations

print("=" * 55)
print("  DATA QUALITY AUDIT")
print("=" * 55)
print(f"\nRaw shape (dates × tickers) : {prices.shape}")

# Drop tickers with NO data at all
all_nan_tickers = prices.columns[prices.isnull().all()].tolist()
prices.drop(columns=all_nan_tickers, inplace=True)
print(f"All-NaN tickers    : {len(all_nan_tickers)} removed → {all_nan_tickers[:10]}")

# Drop tickers with > 10% missing values
MISSING_THRESHOLD = 0.10
missing_frac = prices.isnull().mean()
high_missing  = missing_frac[missing_frac > MISSING_THRESHOLD].index.tolist()
prices.drop(columns=high_missing, inplace=True)
print(f"High-missing (>10%): {len(high_missing)} removed → {high_missing[:10]}")

# Forward-fill then backward-fill residual NaNs (public holidays etc.)
before_fill = prices.isnull().sum().sum()
prices.ffill(inplace=True)
prices.bfill(inplace=True)
after_fill  = prices.isnull().sum().sum()
print(f"Residual NaNs      : {before_fill} → filled → {after_fill} remaining")

# Drop fully-NaN rows (dates)
prices.dropna(how="all", inplace=True)

# Remove near-zero variance tickers
low_var_tickers = prices.columns[prices.std() < 0.01].tolist()
prices.drop(columns=low_var_tickers, inplace=True)
print(f"Near-zero variance : {len(low_var_tickers)} removed")

print(f"\n Clean data shape (dates × tickers) : {prices.shape}")
print(f"  Date range : {prices.index.min().date()} → {prices.index.max().date()}")
print(f"  Tickers retained : {prices.shape[1]}")

clean_tickers = prices.columns.tolist()

  DATA QUALITY AUDIT

Raw shape (dates × tickers) : (753, 503)
All-NaN tickers    : 2 removed → ['Q', 'SNDK']
High-missing (>10%): 5 removed → ['GEHC', 'GEV', 'KVUE', 'SOLV', 'VLTO']
Residual NaNs      : 11 → filled → 0 remaining
Near-zero variance : 0 removed

 Clean data shape (dates × tickers) : (753, 496)
  Date range : 2022-01-03 → 2024-12-31
  Tickers retained : 496


In [24]:
# Part A-4
# Calculate Daily Returns
# Daily Return = (Price_today − Price_yesterday) / Price_yesterday

daily_returns = prices.pct_change().dropna()

print("=" * 55)
print("  DAILY RETURNS")
print("=" * 55)
print(f"\nShape           : {daily_returns.shape}")
print(f"Date range      : {daily_returns.index.min().date()} → {daily_returns.index.max().date()}")
print(f"\nDescriptive statistics (first 5 tickers):")
print(daily_returns.iloc[:, :5].describe().round(5))

# Download S&P 500 INDEX for Beta calculation
index_prices  = yf.download("^GSPC", start=START_DATE, end=END_DATE,
                              auto_adjust=True, progress=False)["Close"]
index_returns = index_prices.pct_change().dropna()
index_returns.name = "SP500"

# Align dates
aligned_dates = daily_returns.index.intersection(index_returns.index)
daily_returns = daily_returns.loc[aligned_dates]
index_returns = index_returns.loc[aligned_dates]

  DAILY RETURNS

Shape           : (752, 496)
Date range      : 2022-01-04 → 2024-12-31

Descriptive statistics (first 5 tickers):
Ticker          A       AAPL       ABBV       ABNB        ABT
count   752.00000  752.00000  752.00000  752.00000  752.00000
mean     -0.00000    0.00059    0.00061    0.00007   -0.00011
std       0.01866    0.01707    0.01386    0.02947    0.01372
min      -0.09665   -0.05868   -0.12566   -0.13425   -0.06544
25%      -0.01083   -0.00844   -0.00619   -0.01554   -0.00798
50%      -0.00011    0.00112    0.00100    0.00021   -0.00019
75%       0.01084    0.00975    0.00817    0.01596    0.00715
max       0.08721    0.08897    0.06361    0.13353    0.07816


In [25]:
# Part A-5
# Calculate Beta for each stock

market_var = index_returns.squeeze().var()    # Variance(R_m) — constant across all stocks

betas = {}
for ticker in clean_tickers:
    stock_ret     = daily_returns[ticker]
    covariance    = stock_ret.cov(index_returns.squeeze())     # Covariance(R_e, R_m)
    betas[ticker] = covariance / market_var          # β = Cov(R_e, R_m) / Var(R_m)

beta_series = pd.Series(betas, name="Beta").sort_values()

print("=" * 55)
print("  BETA VALUES")
print("=" * 55)
print(f"\nBeta statistics:")
print(beta_series.describe().round(4))
print(f"\nStocks with Beta > 1.5  (highly aggressive) : {(beta_series > 1.5).sum()}")
print(f"Stocks with 0.8 ≤ Beta ≤ 1.2 (market-like) : {((beta_series >= 0.8) & (beta_series <= 1.2)).sum()}")
print(f"Stocks with Beta < 0.5  (highly defensive)  : {(beta_series < 0.5).sum()}")
print(f"\nTop 10 highest-Beta stocks:\n{beta_series.nlargest(10).round(4).to_string()}")
print(f"\nTop 10 lowest-Beta stocks:\n{beta_series.nsmallest(10).round(4).to_string()}")



  BETA VALUES

Beta statistics:
count    496.0000
mean       0.9437
std        0.4318
min        0.1332
25%        0.6330
50%        0.9117
75%        1.1557
max        3.2941
Name: Beta, dtype: float64

Stocks with Beta > 1.5  (highly aggressive) : 48
Stocks with 0.8 ≤ Beta ≤ 1.2 (market-like) : 196
Stocks with Beta < 0.5  (highly defensive)  : 72

Top 10 highest-Beta stocks:
CVNA    3.2941
COIN    2.9610
XYZ     2.5448
NVDA    2.2685
APP     2.2619
TTD     2.2371
PLTR    2.1896
MPWR    2.1341
AMD     2.0647
VRT     2.0609

Top 10 lowest-Beta stocks:
GIS    0.1332
CPB    0.1337
SJM    0.1966
HRL    0.2366
LMT    0.2439
JNJ    0.2466
KHC    0.2497
MRK    0.2588
HSY    0.2718
CAG    0.2760


In [26]:
# Part A-6
# Calculate Annual Volatility

# Formula: Annual Volatility = σ_daily × √252
# 252 = conventional number of trading days in a year.

TRADING_DAYS = 252

annual_vol = daily_returns.std() * np.sqrt(TRADING_DAYS)
annual_vol.name = "Annual_Volatility"

print("=" * 55)
print("  ANNUAL VOLATILITY")
print("=" * 55)
print(f"\nAnnual Volatility statistics:")
print(annual_vol.describe().round(4))
print(f"\nTop 10 most volatile stocks:")
print(annual_vol.nlargest(10).apply(lambda x: f"{x*100:.2f}%").to_string())
print(f"\nTop 10 least volatile stocks:")
print(annual_vol.nsmallest(10).apply(lambda x: f"{x*100:.2f}%").to_string())



  ANNUAL VOLATILITY

Annual Volatility statistics:
count    496.0000
mean       0.3143
std        0.1148
min        0.1562
25%        0.2401
50%        0.2850
75%        0.3573
max        1.3410
Name: Annual_Volatility, dtype: float64

Top 10 most volatile stocks:
Ticker
CVNA    134.10%
COIN     95.38%
SMCI     90.20%
APP      78.84%
PLTR     69.09%
XYZ      65.38%
HOOD     65.18%
TTD      64.77%
VRT      62.98%
TSLA     61.30%

Top 10 least volatile stocks:
Ticker
KO       15.62%
JNJ      16.38%
PEP      17.04%
MCD      17.35%
CL       17.37%
BRK-B    17.38%
PG       17.59%
RSG      17.80%
MDLZ     18.50%
KMB      18.56%


In [27]:
# Part A-7
# Combine Metrics & Segment Stocks by Risk Profile

metrics_df = pd.DataFrame({
    "Mean_Daily_Return": daily_returns.mean(),
    "Beta"             : beta_series,
    "Annual_Volatility": annual_vol
})

metrics_df = metrics_df.merge(
    sp500_info[["Ticker","Company","Sector"]].set_index("Ticker"),
    left_index=True, right_index=True, how="left"
)

# Rule-based Risk Segmentation using quantile thresholds
beta_low   = metrics_df["Beta"].quantile(0.33)
beta_high  = metrics_df["Beta"].quantile(0.67)
vol_low    = metrics_df["Annual_Volatility"].quantile(0.33)
vol_high   = metrics_df["Annual_Volatility"].quantile(0.67)

def assign_segment(row):
    b = row["Beta"]
    v = row["Annual_Volatility"]
    if   b <= beta_low  and v <= vol_low:   return "  Conservative"
    elif b >= beta_high and v >= vol_high:  return "  Aggressive"
    elif b <= beta_low  and v >= vol_high:  return "  High-Vol / Low-Beta"
    elif b >= beta_high and v <= vol_low:   return "  Market-Sensitive / Stable"
    else:                                   return "  Balanced / Moderate"

metrics_df["Risk_Segment"] = metrics_df.apply(assign_segment, axis=1)

print("=" * 55)
print("  RISK SEGMENT DISTRIBUTION")
print("=" * 55)
print()
print(metrics_df["Risk_Segment"].value_counts().to_string())
print()

segment_stats = metrics_df.groupby("Risk_Segment")[
    ["Mean_Daily_Return","Beta","Annual_Volatility"]
].mean().round(4)
segment_stats["Mean_Daily_Return"] = segment_stats["Mean_Daily_Return"] * 100
segment_stats["Annual_Volatility"] = segment_stats["Annual_Volatility"] * 100
segment_stats.columns = ["Mean Daily Return (%)", "Beta", "Annual Volatility (%)"]

print("Average metrics per risk segment:")
print(segment_stats.to_string())

print("\n\nSector composition per segment (top-3 sectors):")
for seg in metrics_df["Risk_Segment"].unique():
    top_sectors = (metrics_df.loc[metrics_df["Risk_Segment"] == seg, "Sector"]
                   .value_counts().head(3))
    print(f"\n  {seg}")
    for sector, cnt in top_sectors.items():
        print(f"    {sector:<40s} {cnt:>3d} stocks")

  RISK SEGMENT DISTRIBUTION

Risk_Segment
Balanced / Moderate          249
Aggressive                   119
Conservative                 111
High-Vol / Low-Beta           16
Market-Sensitive / Stable      1

Average metrics per risk segment:
                           Mean Daily Return (%)    Beta  Annual Volatility (%)
Risk_Segment                                                                   
Aggressive                                  0.06  1.5070                  45.33
Balanced / Moderate                         0.04  0.9036                  28.84
Conservative                                0.03  0.4778                  21.57
High-Vol / Low-Beta                         0.05  0.5977                  37.09
Market-Sensitive / Stable                   0.08  1.1474                  25.34


Sector composition per segment (top-3 sectors):

    Balanced / Moderate
    Industrials                               54 stocks
    Financials                                45 stocks
    Health 